In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

from experiments import label_encoder_gender

In [4]:
#Load all the trained model


model = load_model('churn_model.h5')

#Load encoder and scaler
with open('one_hot_encoder.pkl', 'rb') as f:
    label_encoder_geo = pickle.load(f)

with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)



In [68]:
sample_input = {
    "customer_id": 99999999,  # Unique ID not in file
    "credit_score": 580,  # High credit score (typical range 300-850)
    "country": "Spain",  # Not in file (file has France, Spain, Germany based on typical bank data)
    "gender": "Female",  # Alternative to Female
    "age": 55,  # Different age value
    "tenure": 12,  # Years as customer
    "balance": 2500.50,  # Different balance amount
    "products_number": 4,  # Different number of products
    "credit_card": 0,  # Has credit card (1=Yes, 0=No)
    "active_member": 1,  # Active member (1=Yes, 0=No)
    "estimated_salary": 275000000.99 # Different salary

}

In [69]:
geo_encoded = label_encoder_geo.transform([[sample_input['country']]])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['country']))
geo_encoded_df

C:\Development\GenAI_Udemy_Krishnaik\ANN_CLASSIFICATION\ANN_env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,country_Germany,country_Spain
0,0.0,1.0


In [70]:
print(geo_encoded)

[[0. 1.]]


In [71]:
input_df = pd.DataFrame([sample_input])
input_df

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
0,99999999,580,Spain,Female,55,12,2500.5,4,0,1,2.750000e+08


In [72]:
#combining encoded data to df

input_df['gender']


0    Female
Name: gender, dtype: str

In [73]:
input_df["country"] = label_encoder_geo.transform(input_df[["country"]])


In [74]:
input_df["country"]

0    0.0
Name: country, dtype: float64

In [75]:
input_df

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
0,99999999,580,0.0,Female,55,12,2500.5,4,0,1,2.750000e+08


In [76]:
input_df['gender'] = label_encoder_gender.transform(input_df[["gender"]])

C:\Development\GenAI_Udemy_Krishnaik\ANN_CLASSIFICATION\ANN_env\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


In [77]:
input_df['gender']

0    0
Name: gender, dtype: int64

In [78]:
input_df

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
0,99999999,580,0.0,0,55,12,2500.5,4,0,1,2.750000e+08


In [79]:
input_df = pd.concat([input_df.drop('country', axis=1), geo_encoded_df], axis=1)

In [80]:
input_df

,customer_id,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,country_Germany,country_Spain
0,99999999,580,0,55,12,2500.5,4,0,1,2.750000e+08,0.0,1.0


In [81]:
input_df.drop('customer_id', axis=1, inplace=True)

In [82]:
scaler.transform(input_df)

array([[-7.43539782e-01, -1.09499335e+00,  1.53088014e+00,
         2.42782596e+00, -1.17843508e+00,  4.25868381e+00,
        -1.54035103e+00,  9.74816989e-01,  4.77958819e+03,
        -5.79467227e-01,  1.73494238e+00]])

In [83]:
# prediction


prediction = model.predict(scaler.transform(input_df))
prediction


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


array([[0.]], dtype=float32)

In [84]:
if prediction[0][0] > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
